In [6]:
# model_b_start_plus_context.py
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ----------------------------
# Load data & basic config
# ----------------------------
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

# Ensure these columns exist in df_data: start_station_idx, end_station_idx
# and df_data contains ONLY normalized features besides meta columns.
meta_cols = [
    "ride_id", "started_at", "ended_at", "start_station_id", "end_station_id",
    "time_hms_ms", "member_casual"
]
# If you have start/end idx columns already:
assert "start_station_idx" in df_data.columns and "end_station_idx" in df_data.columns, \
       "Necesito start_station_idx y end_station_idx en df_data."

# Build feature matrix (context) — same columns you used at training time
feature_columns = [c for c in df_data.columns if c not in meta_cols + ["start_station_idx", "end_station_idx"]]
X = df_data[feature_columns].values.astype(np.float32)
starts = df_data["start_station_idx"].values.astype(int)
ends = df_data["end_station_idx"].values.astype(int)

# station offset and num_stations (reconstruct same indexing scheme)
station_offset = int(min(df_data["start_station_idx"].min(), df_data["end_station_idx"].min()))
num_stations = int(max(df_data["start_station_idx"].max(), df_data["end_station_idx"].max()) - station_offset + 1)
num_features = X.shape[1]

print("Num stations:", num_stations)
print("Num features:", num_features)
print("station_offset:", station_offset)

# Shift indices to 0..(num_stations-1) if not already
starts_shift = (starts - station_offset).astype(int)
ends_shift   = (ends - station_offset).astype(int)

# ----------------------------
# Train/val/test split (sin stratify)
# ----------------------------
X_train, X_temp, start_train, start_temp, end_train, end_temp = train_test_split(
    X, starts_shift, ends_shift, test_size=0.3, random_state=42
)

X_val, X_test, start_val, start_test, end_val, end_test = train_test_split(
    X_temp, start_temp, end_temp, test_size=0.5, random_state=42
)

print("Train/Val/Test sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])

# ----------------------------
# Optionally compute class weights
# ----------------------------
# For huge class imbalance class_weight helps. Use if you plan to train with fit(..., class_weight=...)
compute_class_weights = True
if compute_class_weights:
    unique_classes_train = np.unique(end_train)
    cw_values = class_weight.compute_class_weight("balanced", classes=unique_classes_train, y=end_train)
    class_weight_dict = {i: cw_values[j] for j, i in enumerate(unique_classes_train)}
else:
    class_weight_dict = None

# ----------------------------
# Build model (Start + Context -> End)
# ----------------------------
# Hyperparams (tune as needed)
embedding_dim = int(min(128, max(32, int(np.ceil(np.sqrt(num_stations))))))  # e.g. sqrt or 64..128
start_embed_dim = embedding_dim
dense_1 = 512
dense_2 = 256
dropout_rate = 0.2
l2_reg = 1e-4

# Inputs
context_input = layers.Input(shape=(num_features,), name="context")
start_input   = layers.Input(shape=(1,), name="start_station_input")

# Embedding for start
start_embed = layers.Embedding(input_dim=num_stations, output_dim=start_embed_dim, name='start_embedding')(start_input)
start_embed = layers.Flatten()(start_embed)  # shape (batch, start_embed_dim)

# Combine
x = layers.Concatenate()([context_input, start_embed])
x = layers.Dense(dense_1, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(dropout_rate)(x)

x = layers.Dense(dense_2, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(dropout_rate)(x)

# Output
end_output = layers.Dense(num_stations, activation='softmax', name='end_station')(x)

model = Model(inputs=[context_input, start_input], outputs=end_output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# ----------------------------
# Callbacks
# ----------------------------
checkpoint_path = "best_model.keras"
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
]

# ----------------------------
# Train
# ----------------------------
batch_size = 1024
epochs = 50

history = model.fit(
    {'context': X_train, 'start_station_input': start_train.reshape(-1, 1)},
    end_train,
    validation_data=({'context': X_val, 'start_station_input': start_val.reshape(-1, 1)}, end_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    class_weight=class_weight_dict,  # optional, helps with imbalance
    verbose=2
)

# ----------------------------
# Evaluation utilities: top-k accuracy and MRR
# ----------------------------
def top_k_accuracy(preds, labels, k=5):
    # preds shape (N, C); labels shape (N,)
    topk = np.argpartition(-preds, kth=k-1, axis=1)[:, :k]  # fast top-k (unordered)
    # for exact ordering, you can use argsort, but slower for large C
    match = [1 if label in topk_row else 0 for topk_row, label in zip(topk, labels)]
    return np.mean(match)

def top_k_accuracy_ordered(preds, labels, k=5):
    topk_ordered = np.argsort(-preds, axis=1)[:, :k]
    match = [1 if label in row else 0 for row, label in zip(topk_ordered, labels)]
    return np.mean(match)

def mrr(preds, labels):
    # Mean Reciprocal Rank
    ranks = []
    order = np.argsort(-preds, axis=1)
    for row_idx, label in enumerate(labels):
        rank_positions = np.where(order[row_idx] == label)[0]
        if rank_positions.size == 0:
            ranks.append(0.0)
        else:
            ranks.append(1.0 / (rank_positions[0] + 1))
    return np.mean(ranks)

# ----------------------------
# Evaluate on test set
# ----------------------------
preds_test = model.predict({'context': X_test, 'start_station_input': start_test.reshape(-1,1)}, verbose=0)
acc = np.mean(np.argmax(preds_test, axis=1) == end_test)
top5 = top_k_accuracy_ordered(preds_test, end_test, k=5)
top10 = top_k_accuracy_ordered(preds_test, end_test, k=10)
mrr_val = mrr(preds_test, end_test)

print("Test accuracy (top-1):", acc)
print("Test top-5:", top5)
print("Test top-10:", top10)
print("Test MRR:", mrr_val)

# ----------------------------
# Save final model
# ----------------------------
model.save("../../models/model_B_start_context/model_B_start_context.keras")

# ----------------------------
# Inference helper functions
# ----------------------------
def predict_single(model, context_vector, start_station_original_id, station_offset):
    """
    context_vector: 1D numpy array with num_features (already normalized in same way as train)
    start_station_original_id: original id (not shifted)
    """
    ctx = context_vector.astype(np.float32).reshape(1, -1)
    start_idx = np.array([int(start_station_original_id) - station_offset]).reshape(1,1)
    preds = model.predict({'context': ctx, 'start_station_input': start_idx}, verbose=0)
    pred_idx = int(np.argmax(preds, axis=1)[0])
    pred_prob = float(np.max(preds, axis=1)[0])
    pred_station_original = pred_idx + station_offset
    return pred_station_original, pred_prob, preds[0]

def predict_batch(model, contexts, start_original_ids, station_offset, batch_size=2048):
    """
    contexts: numpy array shape (N, num_features)
    start_original_ids: array-like shape (N,) with original station ids
    returns: DataFrame with predicted id and prob
    """
    starts_shift = (np.array(start_original_ids).astype(int) - station_offset).reshape(-1,1)
    preds = model.predict({'context': contexts.astype(np.float32), 'start_station_input': starts_shift}, batch_size=batch_size, verbose=0)
    pred_idxs = np.argmax(preds, axis=1)
    pred_probs = np.max(preds, axis=1)
    pred_orig = pred_idxs + station_offset
    df_out = pd.DataFrame({
        "predicted_end_station_idx": pred_orig,
        "predicted_probability": pred_probs
    })
    return df_out

# Example usage of single prediction (using first test row)
example_ctx = X_test[0]
example_start_original = start_test[0] + station_offset  # because start_test is shifted
pred_station, prob, full_vector = predict_single(model, example_ctx, example_start_original, station_offset)
print("Example predicted station:", pred_station, "prob:", prob)


Num stations: 1912
Num features: 27
station_offset: 0
Train/Val/Test sizes: 6657547 1426617 1426618


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ start_station_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ start_embedding     │ (None, 1, 44)     │     84,128 │ start_station_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 27)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 44)        │          0 │ start_embedding[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 71)        │          0 │ context[0][0],    │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 512)       │     36,864 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 256)       │    131,328 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ end_station (Dense) │ (None, 1912)      │    491,384 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 746,776 (2.85 MB)

 Trainable params: 745,240 (2.84 MB)

 Non-trainable params: 1,536 (6.00 KB)

Epoch 1/50

Epoch 1: val_loss improved from None to 20.37057, saving model to best_model.keras
6502/6502 - 140s - 21ms/step - accuracy: 0.0177 - loss: 6.9251 - val_accuracy: 2.1239e-04 - val_loss: 20.3706 - learning_rate: 1.0000e-03
Epoch 2/50

Epoch 2: val_loss did not improve from 20.37057
6502/6502 - 136s - 21ms/step - accuracy: 0.0295 - loss: 5.8709 - val_accuracy: 9.8415e-04 - val_loss: 128.0612 - learning_rate: 1.0000e-03
Epoch 3/50

Epoch 3: val_loss did not improve from 20.37057
6502/6502 - 137s - 21ms/step - accuracy: 0.0320 - loss: 5.3870 - val_accuracy: 3.5749e-05 - val_loss: 54.6456 - learning_rate: 1.0000e-03
Epoch 4/50

Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 4: val_loss did not improve from 20.37057
6502/6502 - 138s - 21ms/step - accuracy: 0.0344 - loss: 5.1090 - val_accuracy: 7.8507e-05 - val_loss: 24.6992 - learning_rate: 1.0000e-03
Epoch 5/50

Epoch 5: val_loss improved from 20.37057 to 9.95119, saving model to best_model.ker

KeyboardInterrupt: 